# ETUDE DE TORSIONS

## Exemples de courbes avec anneaux d'endomorphisme

In [7]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
from sage.libs.libecm import ecmfactor
from sage.misc.search import search
import time

In [8]:
# Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

#l = 5000000029 exemple de calcul d'isogénie, cf article

D%4, p.nbits(), D.nbits() 

(1, 34, 36)

In [9]:
#Exemples Bisson Sutherland

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D%4, p.nbits(), D.nbits() 

(1, 201, 41)

In [10]:
#Exemple 2:

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D%4, p.nbits(), D.nbits() 

(1, 255, 43)

In [11]:
#Exemples Sutherland database

#exemple 1

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D%4, p.nbits(), D.nbits() 

(1, 190, 47)

In [12]:
#exemple 2

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D%4, p.nbits(), D.nbits() 

(1, 255, 54)

## Algorithmes pour etudes de torsions

In [75]:
def etude_torsions(E,p,O,K,Nk_max,Nk_min):

    #On calcule tout les couples degré-torsion de E, dans l'ordre croissant des degrés, jusqu'à trouver une torsion supérieur à (N_max)^2
    #On ignore les couples degré-torsion de E dont la torsion est inférieur à 2N_min
    
    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    rD = f*rK
    wK = (dK + rK)/2
    CE = E.cardinality_pari()
    tracef = E.trace_of_frobenius()
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    s = int((-fm*dK + tracef)/2)
    s2 = (-fm*dK - tracef)/2
    frob = fm*wK + s
    frob2 = -(fm*wK + s2)
    assert (frob^2 - tracef*frob + p) == 0
    assert (frob2^2 - tracef*frob2 + p) == 0

    if dK%4 == 1 :
        a = int((tracef - fm)/2)
    else :
        a = int(tracef/2)
    Nmax = gcd(a-1,fm/f)

    torsions = []
    if Nmax > 2*Nk_min:             #solution à clapoti n'existe pas si N < N1 + N2
        torsions.append([Nmax,1])

    #traces = [tracef]
    #cards = [CE]
    
    frobd = frob
    frob2d = frob2
    q = p
    d = 1
    
    while Nmax < Nk_max^2:
        
        d = d+1
        q = q*p
        frobd = frobd*frob
        frob2d = frob2d*frob2
        tracefd = frobd + frob2d
        
        Dm = tracefd^2 - 4*q
        fm = int(sqrt(Dm/(dK)))
        assert (frobd^2 - tracefd*frobd + q) == 0
        assert (frob2d^2 - tracefd*frob2d + q) == 0
        
        if dK%4 == 1 :
            a = int((tracefd - fm)/2)
        else :
            a = int(tracefd/2)
            
        Nmax = gcd(a-1,fm/f)
        if Nmax > 2*Nk_min:
            torsions.append([Nmax,d])
        #traces.append(tracefd)
        #cards.append(q + 1 - tracefd)
    
    return torsions, d

In [16]:
# Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

#l = 5000000029 exemple de calcul d'isogénie, cf article

torsions, deg_max = etude_torsions(E,p,O,K,isqrt(isqrt((-D))),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(18,
 13,
 [[3, 2],
  [1042, 3],
  [3, 4],
  [18756, 6],
  [3, 8],
  [1042, 9],
  [3, 10],
  [3013, 11],
  [37512, 12],
  [129, 14],
  [1042, 15],
  [3, 16],
  [1069092, 18]])

In [17]:
#Exemples Sutherland

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

torsions, deg_max = etude_torsions(E,p,O,K,isqrt(isqrt((-D))),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(84,
 57,
 [[8, 2],
  [16, 4],
  [22, 5],
  [8, 6],
  [14, 7],
  [96, 8],
  [88, 10],
  [80, 12],
  [1624, 14],
  [22, 15],
  [192, 16],
  [8, 18],
  [176, 20],
  [182, 21],
  [184, 22],
  [1440, 24],
  [22, 25],
  [8, 26],
  [3248, 28],
  [88, 30],
  [384, 32],
  [134, 33],
  [8, 34],
  [154, 35],
  [2960, 36],
  [8, 38],
  [1056, 40],
  [907816, 42],
  [368, 44],
  [22, 45],
  [8, 46],
  [2880, 48],
  [98, 49],
  [88, 50],
  [848, 52],
  [214, 53],
  [872, 54],
  [242, 55],
  [19488, 56],
  [8, 58],
  [4400, 60],
  [2984, 62],
  [182, 63],
  [23808, 64],
  [22, 65],
  [12328, 66],
  [16, 68],
  [1268344, 70],
  [159840, 72],
  [8, 74],
  [3322, 75],
  [16, 76],
  [14, 77],
  [632, 78],
  [2112, 80],
  [8, 82],
  [9078160, 84]])

In [18]:
#Exemple 2:

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()
D

torsions, deg_max = etude_torsions(E,p,O,K,isqrt(isqrt((-D))),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(42,
 22,
 [[32, 2],
  [192, 4],
  [32, 6],
  [1920, 8],
  [352, 10],
  [576, 12],
  [32, 14],
  [3840, 16],
  [32, 18],
  [2112, 20],
  [736, 22],
  [5760, 24],
  [1696, 26],
  [5568, 28],
  [352, 30],
  [238080, 32],
  [134, 33],
  [32, 34],
  [63936, 36],
  [32, 38],
  [105600, 40],
  [4055072, 42]])

In [19]:
#Exemples Sutherland database

#exemple 1

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

torsions, deg_max = etude_torsions(E,p,O,K,isqrt(isqrt((-D))),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(6,
 4,
 [[3, 2],
  [4, 3],
  [3, 4],
  [574097121886238207028063895820770431638971912112893011463831285597096410545328463681656,
   6]])

In [20]:
#exemple 2

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()
D

torsions, deg_max = etude_torsions(E,p,O,K,isqrt(isqrt((-D))),1)
nb_candidats = len(torsions)

deg_max, nb_candidats, torsions

(144,
 80,
 [[3, 2],
  [15, 4],
  [36, 6],
  [15, 8],
  [3, 10],
  [2520, 12],
  [3, 14],
  [15, 16],
  [2052, 18],
  [75, 20],
  [86, 21],
  [3, 22],
  [5040, 24],
  [3, 26],
  [15, 28],
  [36, 30],
  [15, 32],
  [46, 33],
  [3, 34],
  [143640, 36],
  [3, 38],
  [2175, 40],
  [1548, 42],
  [15, 44],
  [3, 46],
  [171360, 48],
  [76053, 50],
  [15, 52],
  [6156, 54],
  [11, 55],
  [15, 56],
  [3, 58],
  [768600, 60],
  [3, 62],
  [86, 63],
  [5295, 64],
  [828, 66],
  [15, 68],
  [3, 70],
  [20971440, 72],
  [9327, 74],
  [15, 76],
  [2844, 78],
  [2175, 80],
  [249, 82],
  [758520, 84],
  [3, 86],
  [1005, 88],
  [2052, 90],
  [911, 91],
  [15, 92],
  [3, 94],
  [342720, 96],
  [3, 98],
  [46, 99],
  [9506625, 100],
  [36, 102],
  [15, 104],
  [86, 105],
  [3, 106],
  [430920, 108],
  [33, 110],
  [15, 112],
  [227, 113],
  [8244, 114],
  [15, 116],
  [3, 118],
  [44578800, 120],
  [1101, 122],
  [15, 124],
  [11205972, 126],
  [5295, 128],
  [3, 130],
  [57960, 132],
  [3, 134],
  [1

# TEST SOLUTIONS KLaPoTi

## myKLPT2

In [22]:
#Extrait du fichier myklpt.py de l'implémentetion Sage de KLaPoTi

def two_squares_none(n):   #CF source two_squares
    """
    Write the integer `n` as a sum of two integer squares if possible;
    otherwise raise a :exc:`ValueError`.

    INPUT:

    - ``n`` -- integer

    OUTPUT: a tuple `(a,b)` of nonnegative integers such that
    `n = a^2 + b^2` with `a <= b`.

    EXAMPLES::

        sage: two_squares(389)
        (10, 17)
        sage: two_squares(21)
        Traceback (most recent call last):
        ...
        ValueError: 21 is not a sum of 2 squares
        sage: two_squares(21^2)
        (0, 21)
        sage: a, b = two_squares(100000000000000000129); a, b                           # needs sage.libs.pari
        (4418521500, 8970878873)
        sage: a^2 + b^2                                                                 # needs sage.libs.pari
        100000000000000000129
        sage: two_squares(2^222 + 1)                                                    # needs sage.libs.pari
        (253801659504708621991421712450521, 2583712713213354898490304645018692)
        sage: two_squares(0)
        (0, 0)
        sage: two_squares(-1)
        Traceback (most recent call last):
        ...
        ValueError: -1 is not a sum of 2 squares

    TESTS::

        sage: for _ in range(100):                                                      # needs sage.libs.pari
        ....:     a = ZZ.random_element(2**16, 2**20)
        ....:     b = ZZ.random_element(2**16, 2**20)
        ....:     n = a**2 + b**2
        ....:     aa, bb = two_squares(n)
        ....:     assert aa**2 + bb**2 == n

    Tests with numpy and gmpy2 numbers::

        sage: from numpy import int16                                                   # needs numpy
        sage: two_squares(int16(389))                                                   # needs numpy
        (10, 17)
        sage: from gmpy2 import mpz
        sage: two_squares(mpz(389))
        (10, 17)

    ALGORITHM:

    See https://schorn.ch/lagrange.html
    """
    n = ZZ(n)

    if n <= 0:
        if n == 0:
            z = ZZ.zero()
            return (z, z)
        return 

    if n.nbits() <= 32:
        from sage.rings import sum_of_squares
        return sum_of_squares.two_squares_pyx(n)

    # Start by factoring n (which seems to be unavoidable)
    F = n.factor(proof=False)

    # First check whether it is possible to write n as a sum of two
    # squares: all prime powers p^e must have p = 2 or p = 1 mod 4
    # or e even.
    for p, e in F:
        if e % 2 and p % 4 == 3:
            return 

    # We run over all factors of n, write each factor p^e as
    # a sum of 2 squares and accumulate the product
    # (using multiplication in Z[I]) in a^2 + b^2.
    from sage.rings.finite_rings.integer_mod import Mod
    a = ZZ.one()
    b = ZZ.zero()
    for p, e in F:
        if e >= 2:
            m = p ** (e // 2)
            a *= m
            b *= m
        if e % 2:
            if p == 2:
                # (a + bi) *= (1 + I)
                a, b = a - b, a + b
            else:  # p = 1 mod 4
                # Find a square root of -1 mod p.
                # If y is a non-square, then y^((p-1)/4) is a square root of -1.
                y = Mod(2, p)
                while True:
                    s = y**((p - 1) / 4)
                    if not s * s + 1:
                        s = s.lift()
                        break
                    y += 1
                # Apply Cornacchia's algorithm to write p as r^2 + s^2.
                r = p
                while s * s > p:
                    r, s = s, r % s
                r %= s

                # Multiply (a + bI) by (r + sI)
                a, b = a * r - b * s, b * r + a * s

    a = a.abs()
    b = b.abs()
    assert a * a + b * b == n
    return (a, b) if a <= b else (b, a)



def trysolve(f, y):
    y = ZZ(y)
    if y <= 0:
        return
    if y.is_pseudoprime():
        return two_squares_none(y)  #KLaPoTi fait appel à Cornacchia ici
    else: 
        return

##def equivalent_prime_ideal(I):
##    Q = I.quaternion_algebra()
##    N0 = I.norm()
##    L = IntegralLattice(I.gram_matrix()).lll().basis_matrix() * I.basis_matrix()
##    bnd = 1
##    while True:
##        for _ in range(5):
##            δ = Q(sum(randrange(-bnd,bnd+1)*v for v in L))
##            N = ZZ(δ.reduced_norm() / N0)
##            if N.is_pseudoprime():
##                break
##        else:
##            bnd += 1
##            continue
##        break
##    print(f'{δ = }')
##    assert δ in I
##    J = I * (δ.conjugate() / N0)
##    del N0
##    print(f'{I = }')
##    print(f'{N = }')
##    return J

def norm_and_generator(I):
    N = ZZ(I.norm())
    O0 = I.left_order()
    bnd = 1
    while True:
        for _ in range(5):
            α = sum(randrange(-bnd,bnd+1)*b for b in I.basis())
            if gcd(α.reduced_norm(), N**2) == N:
                break
        else:
            bnd += 1
            continue
        break
    else:
        assert False
#    print(f'{α = }')
    assert I == O0*N + O0*α
    return N, α

def represent_integer(O0, rhs):
    Q = O0.quaternion_algebra()
    ii,jj,kk = Q.gens()
    if Q.quaternion_order(Q.basis()).discriminant() != 4 * ii**2 * jj**2:
        raise NotImplementedError
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF(1, 0, q)    # x^2 + y^2
    cbnd = isqrt(rhs / 2 / p)
    dbnd = isqrt(rhs / 2 / (p*q))
    if not cbnd or not dbnd:
        print('erreur lN1 trop petit')
        return
    for _ in range(999):
        c = randrange(1,cbnd+1)
        d = randrange(1,dbnd+1)
        rhs1 = rhs - p*nf(c,d)

        sol = trysolve(nf, rhs1)
        if sol is not None:
            a,b = sol
            break
    else:
        print('Pas de solution trouvée')
        return
    γ = Q([a,b,c,d])
#    print(f'{γ = }')
    assert γ in O0
    assert γ.reduced_norm() == rhs
    return γ

def ideal_mod_constraint(N, α, γ):
    ii,jj,kk = α.parent().gens()
    mat = matrix(GF(N), [list(elt) for elt in (γ*jj, γ*kk, α, ii*α, jj*α, kk*α)])
    ker = mat.left_kernel_matrix()
    return next(filter(bool, ker[:,:2].change_ring(ZZ)))  #TODO kernel rank > 1?

def strong_approximation(N, α, C, D, rhs):
    ii,jj,kk = α.parent().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])    # x^2 + y^2
    rhs1 = Mod(rhs, N) / p / nf(C,D)
    if not rhs1.is_square():
        l = next(l for l in rhs.prime_divisors() if not Mod(l,N).is_square())
        rhs //= l
        rhs1 //= l
        assert rhs1.is_square()
    λ = ZZ(rhs1.sqrt())    #lambda n'est pas le bon carré ? approx ?
    print(f'{λ = }')
    λC,λD = (λ * vector(GF(N), (C,D))).change_ring(ZZ)

    x,y,z,t = polygens(ZZ, 'x,y,z,t')
    eqn = rhs - nf(N*x,N*y) - p*nf(λC+N*z, λD+N*t)
    print(f'{eqn =}')
    print(f'{eqn(0,0,z,t) =}')
    assert eqn % N == 0
    eqn1 = eqn // N % N
    U, V, W = eqn1[z], eqn1[t], -eqn1.constant_coefficient()
    assert eqn1 == U*z + V*t - W

    # Petit-Smith
    lat = matrix([[U,0,1,0],[V,0,0,1],[-W,1,0,0]]).stack(N*identity_matrix(4))
    scal = diagonal_matrix([N**2, N, 1, 1])
    for row in matrix(ZZ, filter(bool, (lat * scal).LLL() * ~scal)):
#        print(row)
        if row[1] < 0:
            row = -row
        if not row[0] and row[1] == 1:
            sol0 = row[2:]
            break
    else:
        assert False, 'should never happen'
    import fpylll
    mat = matrix(filter(bool, matrix([[V,-U],[N,0],[0,N]]).LLL()))
    assert mat.dimensions() == (2, 2)
    lat = fpylll.IntegerMatrix(2, 2)
    for i,row in enumerate(mat):
        for j,c in enumerate(row):
            lat[i,j] = c
    gso = fpylll.GSO.Mat(lat)  #TODO lengths are slightly off when q>1
    gso.update_gso()
    cnt = 10
    seen = set()
    count_negatif = 0   #compteur ajouté
    while True:
        enum = fpylll.Enumeration(gso, cnt, fpylll.EvaluatorStrategy.BEST_N_SOLUTIONS)
        rs = enum.enumerate(0, 2, N**2, 0, tuple(mat.solve_left(sol0)))
        for r in rs:
            z,t = sol0 - vector(ZZ,r[1])*mat
            assert eqn1(0,0,z,t) % N == 0
            if (z,t) in seen:
                continue
            seen.add((z,t))
            assert eqn(0,0,z,t) % N**2 == 0
            rhs2 = ZZ(eqn(0,0,z,t)) // N**2
            print(f'{rhs2=}')
            print(rhs2.factor())
            diff = p*nf(λC+N*z, λD+N*t)
            #print(f'{diff =}')
            marge_disc = (diff / p^3).numerical_approx()
            print(f'{marge_disc =}')
            #print(f'{rhs =}')
            #print(f'{λ =}')
            print(f'{z =}', f'{t = }')
            if rhs2 <= 0:
                count_negatif = count_negatif + 1
            else:
                count_negatif = 0
            assert count_negatif < 10
            sol = trysolve(nf, rhs2)          #rhs2 somme de deux carré? positif ? 
            if sol is not None:
                x,y = sol
                break
        else:
            if len(rs) < cnt:
                raise NotImplementedError
            cnt *= 2
            continue
        break

    γ = (λC*jj + λD*kk) + N*(x + y*ii + z*jj + t*kk)
    print(f'{γ = }')
    print(f'{γ.reduced_norm() = }')
    assert γ.reduced_norm() == rhs
    return γ

In [23]:
#Pour adapter l'algorithme aux second membres qui ne sont pas des puissances de deux, 
#On a besoin de factoriser au hasard un entier N = N_1N_2


def liste(facto):
    #On converti un résultat de la méthode .factor() en liste. 
    k = len(facto)
    facto_liste = []
    for i in range(k):
        facto_liste.append([facto[i][0], facto[i][1]])
    return facto_liste

def rand_facto(N,factoN,D,l):

    #On suppose N non premier, factoN est sa décomposition en facteurs premiers.
    #Le discriminant est donné positif
    
    k = len(factoN)
    facto1 = liste(factoN)
    N1 = N
    N2 = 1
        
    liste_indice = [0 .. k-1]
    while N2 <= D^3 and k>0:  #Taille de N2 donnée par l'algorithme de Petit et Smith
        i = liste_indice[randint(0,k-1)]
        prime = facto1[i][0]
        exp = facto1[i][1]
        if exp > 0:
            N2 = N2*prime
            facto1[i][1] = exp-1
            N1 = N1 // prime
        else:
            liste_indice.remove(i)
        k = len(liste_indice)
    
    assert N == N1*N2
    
    if l*N1 > 3*D and N2 > D^3:   #Taille de N1 donnée par reprensent_integer
        return N1, N2    
    else:
        return rand_facto(N,factoN,D,l)

def klpt(I, N, factoN):
    print(f'{N = }')
    ii,jj,kk = I.quaternion_algebra().gens()
    q, p = ZZ(-ii**2), ZZ(-jj**2)
    nf = BinaryQF([1, 0, q])  #Peut-être jj plutôt ??
    gcdN = N.gcd(p)
    
    FactoG = gcdN.factor(proof=False)
    for p, e in FactoG:
        if e % 2 and p % 4 == 3:
            print(FactoG)
            raise ValueError('Gcd bloquant Cornacchia')
            
    while True:   #Necesaire ?

        if not ZZ(I.norm()).is_pseudoprime():
            raise NotImplementedError         #On suppose I de norme premier, quitte a faire une equivalence. 
#            I = equivalent_prime_ideal(I)

        l,α = norm_and_generator(I)   
        print(l)
        O0 = I.left_order()
        abs_disc = O0.discriminant()   #discriminant positif
            
        test = True
        while test:
            N1, N2 = rand_facto(N,factoN,abs_disc,l)   #TODO : Tester si la facto est déjà vue ?
            assert N1*N2 == N
            print('Tentative repinteger')
            print(f'{N1 = }')
            print(f'{N2 = }')
            γ = represent_integer(O0, l*N1)
            if γ is not None:
                test = False
        print('Marge erreur sur N2', (N2/(abs_disc)^3).numerical_approx())
        print('gcd N2 disc', N2.gcd(abs_disc))
        
        C,D = ideal_mod_constraint(l, α, γ)
        print(f'{C = }')
        print(f'{D = }')
        if l.divides(nf(C,D)):
            raise NotImplementedError('bad')

        µ = strong_approximation(l, α, C, D, N2)
        
        return γ * µ

## Contexte pour KLaPoTi

In [25]:
def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier : Cf notes de cours Biasse
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l) #Lien avec Seysen ? Prendre le min des deux racines vu dans N ? (Optionnel)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) #Forme donnée dans le cours de Biasse
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"





def sol_KLPT(N,factoN,aa):

    #Utiliser KLPT pour résoudre l'equation définie par N et aa
    while True :
    
        l, α = aa.gens_two()
        dK = α.parent().number_field().discriminant()
        Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
        assert (α[0] + α[1]*j).reduced_norm() == α.norm()
        assert (α[0] + α[1]*j).reduced_trace() == α.trace()
        assert j.reduced_norm() == rK.norm() and j.reduced_trace() == rK.trace()
        r = α.parent().number_field().gen()
        assert (r+1)/2 in O   #Necessaire ? ordre de discriminant = 1 mod 4 ?
        OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
        I = OO*l + OO*(α[0] + α[1]*j)
        print(f'{I = }')


        elt = klpt(I,N,factoN)
        assert elt in I
        print(f'{elt = }', '| norm:', elt.reduced_norm().factor())
    

        b = elt[0] + elt[2]*r
        c = elt[1] + elt[3]*r
        #while b and c and b/2 in aa and c/2 in aa:  # can we avoid this a priori in KLPT?
            #b /= 2
            #c /= 2
        print(f'{b = }')
        print(f'{c = }')
        assert b in aa
        assert c in aa
        bb = O.ideal([g*b.conjugate()/aa.norm() for g in aa.gens()])
        cc = O.ideal([g*c.conjugate()/aa.norm() for g in aa.gens()])
        Nb = bb.norm()
        Nc = cc.norm()
        print(f'{Nb = }')#.factor())
        print(f'{Nc = }')#.factor())
        if ZZ(Nb + Nc) == N:
            break
        else:
            print('Erreur sur Nb, Nc')
    if Nb.gcd(Nc) == 1:
        print('solution premier entre eux')
        return Nb, Nc, 1
    else :
        print('solution avec gcd>1')
        return Nb, Nc, Nb.gcd(Nc)
        
        

In [30]:
def reduction_ideal_premier(aa):
    #Trouve un idéal équivalent à aa, de norme un nombre premier. 
    qa = aa.quadratic_form()
    ra = qa.reduced_form()
    lr = ra.small_prime_value()
    aar = ideal_de_norme(lr,f,D)
    if aar.is_equivalent(aa):
        return aar
    else: 
        return aar.conjugate()


In [38]:
# Exemple de Jao "small"

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

CE = E.cardinality_pari()
tracef = E.trace_of_frobenius()
Dm = tracef^2 - 4*p
fm = sqrt(Dm/(K.discriminant()))

#l = 5000000029 exemple de calcul d'isogénie, cf article

D%4, p.nbits(), D.nbits() 

(1, 34, 36)

In [39]:
l =randint(10^20, 10^21)
aa = ideal_de_norme(l,f,D)
aa = reduction_ideal_premier(aa)
aa.norm()

11

In [41]:
l, α = aa.gens_two()
dK = α.parent().number_field().discriminant()
Quat.<i,j,k> = QuaternionAlgebra(-1, dK)
OO = Quat.quaternion_order([1, i, (1+j)/2, (i+k)/2])
I = OO*l + OO*(α[0] + α[1]*j)
Q = OO.quaternion_algebra()
ii,jj,kk = Q.gens()
assert Q.quaternion_order(Q.basis()).discriminant() == 4 * ii**2 * jj**2

q, p = ZZ(-ii**2), ZZ(-jj**2)

In [53]:
N = 2^((D^4).nbits()) + randint(1,2^10)   #Taille de N pour avoir un rhs2 positif : log(D^(3,5),2) = 52
# Exemple qui fonctionne 2^180 :N = 1532495540865888858358347027150309183618739122183602389
# Exemple qui fonctionne 2^130 :N = 1361129467683753853853498429727072846691
factoN = N.factor()
print(factoN)
print(p.factor())
sol_KLPT(N,factoN,aa)

692059 * 1804657 * 65565671 * 34042001552510246007277
5 * 7 * 11 * 13 * 73 * 109 * 971
I = Fractional ideal (11/2 + 1/2*j, 11/2*i + 1/2*k, j, k)
N = 2787593149816327892691964784081045188248521
11


RecursionError: maximum recursion depth exceeded

# TEST SOLUTION Clapoti-PEGASIS

## Algorithmes Resolution d'équations Clapoti-Pegasis

In [77]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
import time

#Il sera nécessaire d'utiliser l'algorithme d'etude de torsions énoncé précédemment.

def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier.
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) 
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

def ideal_to_element(a,L,O):
    
    #On suppose a equivalent à L. On cherche alpha dans L tel que a = ( (alpha)bar / N(L))L.
    
    assert a.is_equivalent(L)
    B = a*(L.conjugate())
    alphabar = (B.gens_reduced())[0]
    alpha = alphabar.conjugate()
    assert NumberFieldOrderIdeal(O,alphabar) == B
    assert alpha in L
    return alpha

def element_to_ideal(alpha,L,O):
    
    #On suppose alpha dans L et on calcule l'idéal équivalent associé a = ( (alpha)bar / N(L))L.
    
    assert alpha in L
    alphabar = alpha.conjugate()
    B = NumberFieldOrderIdeal(O,alphabar)
    a2 = B*L
    g1, g2 = a2.gens_reduced()
    a = NumberFieldOrderIdeal(O,[g1/(L.norm()), g2/(L.norm())])
    assert a.is_equivalent(L)
    return a

def liste_premiers_splits(D,borne_B,fm,p):

    #On établie une liste de nombres premier majorés par borne_B, décomposé dans O_D, premiers avec f_m et p. 
    
    liste_B = []
    i = 2
    while i < borne_B :
        if kronecker(D,i) == 1 and i.gcd(fm*p) == 1:
            liste_B.append(i)
        i = next_prime(i)
    return liste_B

def make_liste_ideq(L,O,m,liste_B,CE):

    # On établie une liste d'idéaux équivalent à L, de normes minimales, donc on calcule la partie friable de la norme.  
    # paramètre m : Le nombre d'idéaux que l'on teste 2m^2 + m. 
    # On ne retient que les idéaux de normes premier avec CE.
    # Si un idéal équivalent de norme friable est trouvé, on s'arrête.

    qL = L.quadratic_form()
    rL = qL.reduced_form()
    RL = NumberFieldOrderIdeal(O,rL)
    
    rK = O.number_field().gens()[0]
    f = O.conductor()
    alpha = rL[0]
    beta = (-rL[1] + f*rK)/2   #N'utilise pas LLL, contrairement à l'implémentation sage de Pegasis.
    
    liste_ideq = []
    Nk_max = 1
    Nk_min = -(O.discriminant())
    ideal_friable = False
    
    for x in [0 .. m]:   
        for y in [-m .. m]:
            if (x != 0 or y > 0):   
                gamma = x*alpha + y*beta    #Pour ne pas creer gamma et -gamma, on evite les cas x < 0, ou (x = 0 et y <= 0), 
                
                if gamma == 0:
                    raise ValueError('erreur base courte liée') #n'est pas censé arriver. 
                    
                I = element_to_ideal(gamma,RL,O)
                NI = I.norm()
                if NI.gcd(CE) != 1:
                    continue
                    
                Nk = NI
                Ne = 1
                Ne_facto = []
                for p in liste_B:
                    exp = 0
                    while Nk%p == 0:
                        exp = exp + 1
                        Nk = Nk/p
                        Ne = Ne*p
                    if exp > 0:
                        Ne_facto.append([p,exp])
                        
                assert NI == Nk*Ne
                
                if Nk == 1:
                    ideal_friable = True
                    return [[I,Nk,Ne,Ne_facto]], Nk, Nk, ideal_friable
                    
                liste_ideq.append([I,Nk,Ne,Ne_facto])
                if Nk > Nk_max:
                    Nk_max = Nk
                if Nk < Nk_min:
                    Nk_min = Nk
                    
    return liste_ideq, Nk_max, Nk_min, ideal_friable

def coin_equation(N1,N2,N):
    
    #On veut résoudre uN1 + vN2 = N dans NN, avec uN1 gcd vN2 == 1 et N divise Nmax
    
    d,u0,v0 = xgcd(N1,N2)
    assert d == 1
    
    if u0 > 0:
        u, v, N, sol = coin_equation(N2,N1,N)
        return v, u, N, sol

    #On suppose u0 <= 0
        
    ku = int(((-u0)*N) // N2) + 1  #erreur de int, etrange ?
    kv = int((v0*N)//N1)
    u = u0*N + ku*N2
    v = v0*N - ku*N1
    assert u*N1 + v*N2 == N

    sol = False
    if ku > kv :
        #print('echec equation dans NN')
        return u, v, N, sol
        
    nb_sols = kv - ku + 1

    while sol == False and nb_sols >= 0:
        duv = u.gcd(v)
        if N%duv != 0:
            nb_sols = nb_sols - 1
            continue
        if ((u/duv)*N1).gcd((v/duv)*N2) == 1:
            u = u/duv
            v = v/duv
            N = N/duv
            break
        u = u + N2
        v = v - N1
        nb_sols = nb_sols - 1

    assert u*N1 + v*N2 == N
    if (u*N1).gcd(v*N2) == 1 and v > 0 and u > 0:
        sol = True
            
    return u, v, N, sol


def clapoti_equation(torsions,liste_ideq):
    k_id = len(liste_ideq)
    sols = []
    sols_ext = []
    for i in [0 .. k_id-1]:
        b_prep = liste_ideq[i]
        b, N1, M1, factoM1 = b_prep
        
        for j in [i .. k_id-1]:
            c_prep = liste_ideq[j]
            c, N2, M2, factoM2 = c_prep
            
            if N1.gcd(N2) != 1:
                #print(N1,N2,'echec N1 N2 pas premier entre eux')
                continue

            for Nmax, ext in torsions :
                sol_ext = False
                
                Gcd1 = Nmax.gcd(M1)
                Nmax_loc = Nmax/Gcd1
                Gcd2 = Nmax_loc.gcd(M2)
                Nmax_loc = Nmax_loc/Gcd2
                
                Nmin = N1+N2   
                if Nmin > Nmax_loc :
                    continue
                
                #print('Tentative N =',Nmax_loc, 'N1 =', N1, 'N2 =',N2)
                u,v,N,sol = coin_equation(N1,N2,Nmax_loc)
                if sol:
                    #print('solution :',u,'*',N1,'+',v,'*',N2,'=',N)
                    sols.append([u,N1,v,N2,N,i,j])
                    sol_ext = True
                #else:
                    #print(N,N1,N2,'echec equation')
                if sol_ext == True and ext not in sols_ext:
                    sols_ext.append(ext)

    return sols, sols_ext

In [72]:
def first_solutions_clapoti(prime_max,m_id,l,E,K,O):

    # On calcule L un idéal de O de norme l. 
    # Renvoie une solution de l'équation de Clapoti-Pegasis (E,L,prime_max) de degré minimal.  
    # De plus renvoie deg_max est le degré maximale considéré lors de l'etude de la torsion.
    # Enfin renvoie ideal_friable = True si un idéal équivalent friable a été trouvé.
    

    rK = K.gens()[0]
    dK = K.discriminant()
    f = O.conductor()
    D = (f^2)*dK
    CE = E.cardinality_pari()
    p = E.base_ring().characteristic()
    tracef = p + 1 - CE   #E définie sur Fp
    Dm = tracef^2 - 4*p
    fm = sqrt(Dm/(K.discriminant()))
    
    L = ideal_de_norme(l,f,D)
    
    resultat = []
    liste_B = liste_premiers_splits(D,prime_max,fm,p)
    liste_ideq, Nk_max, Nk_min, ideal_friable = make_liste_ideq(L,O,m_id,liste_B,CE)

    if ideal_friable:
        return [], liste_ideq, 1, ideal_friable
        
    torsions_candidats, deg_max = etude_torsions(E,p,O,K,Nk_max,Nk_min)
    for torsion in torsions_candidats:
        sols, sols_ext = clapoti_equation([torsion],liste_ideq)
        if len(sols)>0:
            resultat.append(torsion)
            resultat.append(sols)
            break
    return resultat, liste_ideq, deg_max, ideal_friable

def test_solutions_clapoti(prime_max,m_id,l,E,K,O):

    #Applique l'algorithme first_solution_clapoti en faisant varier la borne des nombres premiers entre 1 et prime_max.
    
    resultat = []
    Bp = 1
    while Bp <= prime_max:
        first, liste_ideq, deg_max, ideal_friable = first_solutions_clapoti(Bp,m_id,l,E,K,O)
        if ideal_friable:
            resultat.append([Bp,deg_max,len(liste_ideq),first[0],1])
        else:
            resultat.append([Bp,deg_max,len(liste_ideq),first[0],len(first[1])])
        Bp = next_prime(Bp)
    return resultat

def stat_solutions_clapoti(prime_max, m_id, nb_essais,E,K,O): 

    #Applique l'algorithme first_solution_clapoti en faisant varier les normes l de dépard, un nombre de fois éale à nb_essais. 
    #l est pris au hasard entre 10^20 et 10^21.

    print('prime_max :', prime_max)

    moy_temps = 0
    moy_deg = 0
    moy_id = 0
    nb_id_friable = 0
    max_deg = 0
    min_deg = 0
    D = O.discriminant()

    print('Nombre essais', nb_essais)

    non_friable = 0

    while non_friable < 20 and nb_id_friable < 200:

        l = next_prime(randint(10^20, 10^21))
        while kronecker(D,l) != 1:
            l = next_prime(l)
        debut = time.time()
        first, liste_ideq, deg_max, ideal_friable = first_solutions_clapoti(prime_max,m_id,l,E,K,O)
        temps = time.time() - debut

        if ideal_friable:
            print('friable', nb_id_friable)
            nb_id_friable += 1
        else:
            non_friable += 1
            print('non_friable', non_friable)
            deg_sol = first[0][1]
            moy_id = moy_id + len(liste_ideq)
            moy_deg = moy_deg + deg_sol
            moy_temps = moy_temps + temps
            if deg_sol > max_deg:
                max_deg = deg_sol
            if deg_sol < min_deg or min_deg == 0:
                min_deg = deg_sol

    print('temps moyen first solution :',moy_temps/non_friable)
    print('degrés moyen de first solution :', moy_deg/non_friable)
    print('dégrés maximal parmis les solutions :', max_deg)
    print('degrés minimal parmis les solutions :', min_deg)
    print('Nb idéaux théoriques :', m_id*m_id*2 + m_id)
    print('Nb idéaux retenus en moyenne :', moy_id/non_friable)
    print('Nb de tentatives ignorées (idéal friable) :', nb_id_friable)
    print('Fréquence idéaux friables :', nb_id_friable/(nb_id_friable + non_friable) )
    print('\n')

    return 

In [73]:
qL = BinaryQF([9,5,9])
D = qL.discriminant()
D.factor()
K.<rK> = QuadraticField(D)
O = K.maximal_order()
L = NumberFieldOrderIdeal(O,qL)
L.norm()

9

## Etudes des exemples

In [76]:
#Exemple de Jao "small"

print('Exemple JaoSmall')

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  # choix fait dans Pegasis

prime_max = 370           # taille max polynome modulaire sagemath.

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

Exemple JaoSmall
Discriminant = -38669866235
prime_max : 370
Nombre essais 20
friable 0
non_friable 1
non_friable 2
friable 1
friable 2
non_friable 3
non_friable 4
non_friable 5
friable 3
non_friable 6
non_friable 7
non_friable 8
non_friable 9
non_friable 10
non_friable 11
non_friable 12
friable 4
friable 5
non_friable 13
non_friable 14
non_friable 15
non_friable 16
non_friable 17
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 4.712559831142426
degrés moyen de first solution : 171/10
dégrés maximal parmis les solutions : 60
degrés minimal parmis les solutions : 3
Nb idéaux théoriques : 78
Nb idéaux retenus en moyenne : 176/5
Nb de tentatives ignorées (idéal friable) : 6
Fréquence idéaux friables : 3/13




In [88]:
prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max : 1000
Nombre essais 20
friable 0
friable 1
friable 2
friable 3
friable 4
friable 5
non_friable 1
friable 6
non_friable 2
friable 7
friable 8
friable 9
non_friable 3
friable 10
friable 11
friable 12
friable 13
friable 14
non_friable 4
friable 15
friable 16
friable 17
friable 18
friable 19
friable 20
friable 21
non_friable 5
friable 22
friable 23
friable 24
friable 25
friable 26
friable 27
non_friable 6
friable 28
friable 29
friable 30
friable 31
non_friable 7
friable 32
friable 33
friable 34
friable 35
non_friable 8
non_friable 9
friable 36
friable 37
friable 38
friable 39
friable 40
non_friable 10
friable 41
non_friable 11
friable 42
non_friable 12
friable 43
non_friable 13
friable 44
non_friable 14
friable 45
non_friable 15
non_friable 16
friable 46
non_friable 17
friable 47
friable 48
friable 49
non_friable 18
friable 50
friable 51
friable 52
friable 53
friable 54
friable 55
friable 56
friable 57
non_friable 19
friable 58
friable 59
friable 60
non_friable 20
temps moyen fi

In [8]:
#EXEMPLES Bisson Sutherland

print('Exemple BS1')

#Exemple 1:

p = 1606938044258990275550812343206050075546550943415909014478299
Fp = GF(p)
E = EllipticCurve(Fp,[-3,660897170071025494489036936911196131075522079970680898049528])

K.<rK> = QuadraticField(-7)
f = 524287   #trouver par algo de Sutherland
O = K.order(f*(1 + rK)/2)

D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

Exemple BS1
Discriminant = -1924138008583


In [ ]:
prime_max = 370           # taille max polynome modulaire sagemath. 

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

Exemple BS1
Discriminant = -1924138008583
prime_max : 370
Nombre essais 20
non_friable 1
non_friable 2
friable 0
friable 1
non_friable 3
friable 2
friable 3
friable 4
friable 5
non_friable 4
friable 6
non_friable 5
friable 7
non_friable 6
friable 8
non_friable 7
friable 9
non_friable 8
friable 10
friable 11
friable 12
non_friable 9
non_friable 10
friable 13
friable 14
ERROR: removing wrong instance of Gen
non_friable 11
non_friable 12
friable 16
non_friable 13


/home/maxime/sage/sage/local/var/lib/sage/venv-python3.12.5/lib/python3.12/site-packages/ipykernel/iostream.py:680: RuntimeWarning: cypari2 leaked 136159590067312 bytes on the PARI stack
  buf = self._rotate_buffer()


SignalError: Segmentation fault

Exception ignored in: 'cypari2.stack.remove_from_pari_stack'
Traceback (most recent call last):
  File "cypari2/gen.pyx", line 276, in cypari2.gen.Gen.__str__
  File "cypari2/gen.pyx", line 241, in cypari2.gen.Gen.__repr__
cysignals.signals.SignalError: Segmentation fault


ERROR: removing wrong instance of Gen


SignalError: Segmentation fault

Exception ignored in: 'cypari2.stack.remove_from_pari_stack'
Traceback (most recent call last):
  File "cypari2/gen.pyx", line 276, in cypari2.gen.Gen.__str__
  File "cypari2/gen.pyx", line 241, in cypari2.gen.Gen.__repr__
cysignals.signals.SignalError: Segmentation fault


friable 15
ERROR: inconsistent avma when removing Gen from PARI stack
Expected: 0x7bd61fec9100
Actual:   0x7bd61fffadb0
non_friable 14
non_friable 15
non_friable 16
friable 17
friable 18
friable 19
friable 20
non_friable 17
friable 21
friable 22
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 25.48492435216904
degrés moyen de first solution : 293/5
dégrés maximal parmis les solutions : 84
degrés minimal parmis les solutions : 36
Nb idéaux théoriques : 465
Nb idéaux retenus en moyenne : 4540/43
Nb de tentatives ignorées (idéal friable) : 23
Fréquence idéaux friables : 23/43




In [9]:
prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

prime_max : 1000
Nombre essais 20
friable 0
non_friable 1
non_friable 2
friable 1
friable 2
friable 3
friable 4
friable 5
friable 6
friable 7
friable 8
friable 9
friable 10
friable 11
friable 12
friable 13
friable 14
friable 15
friable 16
friable 17
friable 18
non_friable 3
friable 19
friable 20
friable 21
friable 22
friable 23
friable 24
friable 25
friable 26
friable 27
friable 28
friable 29
friable 30
friable 31
friable 32
friable 33
friable 34
non_friable 4
friable 35
friable 36
friable 37
friable 38
friable 39
friable 40
friable 41
friable 42
friable 43
friable 44
friable 45
friable 46
friable 47
friable 48
friable 49
friable 50
friable 51
friable 52
friable 53
friable 54
friable 55
friable 56
friable 57
friable 58
friable 59
friable 60
non_friable 5
friable 61
non_friable 6
friable 62
friable 63
friable 64
friable 65
friable 66
friable 67
friable 68
friable 69
friable 70
friable 71
friable 72
friable 73
friable 74
friable 75
friable 76
friable 77
friable 78
friable 79
friable 80
f

/home/maxime/sage/sage/local/var/lib/sage/venv-python3.12.5/lib/python3.12/json/encoder.py:200: RuntimeWarning: cypari2 leaked 124507666617736 bytes on the PARI stack
  chunks = self.iterencode(o, _one_shot=True)


friable 199
temps moyen first solution : 27.739855368932087
degrés moyen de first solution : 140/3
dégrés maximal parmis les solutions : 112
degrés minimal parmis les solutions : 28
Nb idéaux théoriques : 465
Nb idéaux retenus en moyenne : 105
Nb de tentatives ignorées (idéal friable) : 200
Fréquence idéaux friables : 50/53




In [4]:
#Exemple 2:

print('Exemple BS2')

p = 50272551883931021408091448710235646749904660980498576680086699865431843568847
Fp = GF(p)
E = EllipticCurve(Fp,[-3,14262957895783764742987524732821199570860243293007735537575027051453663494306])

K.<rK> = QuadraticField(-7)
f = 852857
O = K.order(f*(1 + rK)/2)


D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

Exemple BS2
Discriminant = -5091555437143


In [ ]:
prime_max = 370           # taille max polynome modulaire sagemath. 

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

Exemple BS2
Discriminant = -5091555437143
prime_max : 370
Nombre essais 20
non_friable 1
friable 0
non_friable 2
friable 1
non_friable 3
friable 2
non_friable 4
friable 3
friable 4
non_friable 5
non_friable 6
friable 5
non_friable 7
non_friable 8
non_friable 9
non_friable 10
non_friable 11
friable 6
friable 7
friable 8
non_friable 12
friable 9
non_friable 13
friable 10
friable 11
friable 12
non_friable 14
non_friable 15
friable 13
non_friable 16
non_friable 17
friable 14
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 30.744378793239594
degrés moyen de first solution : 317/10
dégrés maximal parmis les solutions : 84
degrés minimal parmis les solutions : 8
Nb idéaux théoriques : 528
Nb idéaux retenus en moyenne : 4664/35
Nb de tentatives ignorées (idéal friable) : 15
Fréquence idéaux friables : 3/7




In [10]:
prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max : 1000
Nombre essais 20
friable 0
friable 1
friable 2
friable 3
friable 4
friable 5
friable 6
friable 7
friable 8
friable 9
friable 10
friable 11
friable 12
non_friable 1
friable 13
friable 14
friable 15
friable 16
friable 17
friable 18
friable 19
friable 20
friable 21
friable 22
friable 23
friable 24
friable 25
friable 26
friable 27
friable 28
friable 29
friable 30
friable 31
friable 32
friable 33
friable 34
friable 35
friable 36
non_friable 2
friable 37
friable 38
friable 39
friable 40
non_friable 3
friable 41
friable 42
friable 43
friable 44
friable 45
friable 46
friable 47
non_friable 4
friable 48
friable 49
friable 50
non_friable 5
non_friable 6
friable 51
friable 52
friable 53
friable 54
friable 55
friable 56
friable 57
friable 58
friable 59
friable 60
friable 61
friable 62
friable 63
friable 64
friable 65
friable 66
friable 67
friable 68
friable 69
friable 70
friable 71
friable 72
non_friable 7
friable 73
friable 74
friable 75
non_friable 8
friable 76
friable 77
friabl

In [4]:
#EXEMPLES Sutherland database

#exemple 1

print('Exemple Sutherland1')

p = 1317860422843322160610398725225958731902944552925978150597
Fp = GF(p)
E = EllipticCurve(Fp,[-3,154344787563346744370152153588767287709323583533485442048])

K.<rK> = QuadraticField(-102197306669747)
O = K.maximal_order()

#Creer par CM. On a f = 1 automatique !
D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

Exemple Sutherland1
Discriminant = -102197306669747


In [ ]:
prime_max = 370           # taille max polynome modulaire sagemath. 

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

Exemple Sutherland1
Discriminant = -102197306669747
prime_max : 370
Nombre essais 20
friable 0
non_friable 1
friable 1
non_friable 2
non_friable 3
non_friable 4
non_friable 5
non_friable 6
friable 2
friable 3
non_friable 7
friable 4
non_friable 8
non_friable 9
friable 5
friable 6
friable 7
non_friable 10
non_friable 11
friable 8
non_friable 12
non_friable 13
non_friable 14
non_friable 15
non_friable 16
non_friable 17
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 48.30552614927292
degrés moyen de first solution : 6
dégrés maximal parmis les solutions : 6
degrés minimal parmis les solutions : 6
Nb idéaux théoriques : 406
Nb idéaux retenus en moyenne : 420
Nb de tentatives ignorées (idéal friable) : 9
Fréquence idéaux friables : 9/29




In [5]:
prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20, E,K,O)

prime_max : 1000
Nombre essais 20
friable 0
friable 1
friable 2
friable 3
non_friable 1
friable 4
non_friable 2
non_friable 3
friable 5
friable 6
friable 7
friable 8
friable 9
friable 10
non_friable 4
non_friable 5
friable 11
non_friable 6
friable 12
friable 13
non_friable 7
friable 14
friable 15
friable 16
friable 17
friable 18
friable 19
friable 20
non_friable 8
non_friable 9
friable 21
friable 22
friable 23
friable 24
friable 25
friable 26
friable 27
non_friable 10
non_friable 11
friable 28
friable 29
friable 30
non_friable 12
friable 31
friable 32
friable 33
non_friable 13
friable 34
friable 35
friable 36
friable 37
friable 38
non_friable 14
friable 39
friable 40
friable 41
friable 42
friable 43
non_friable 15
non_friable 16
friable 44
friable 45
friable 46
friable 47
non_friable 17
friable 48
friable 49
friable 50
friable 51
friable 52
friable 53
friable 54
non_friable 18
non_friable 19
friable 55
friable 56
friable 57
non_friable 20
temps moyen first solution : 48.17217794656754


In [6]:
#exemple 2

print('Exemple Sutherland2')

p = 28948022309329048855892746252171992875431396939874100252456123922623314798263
Fp = GF(p)
E = EllipticCurve(Fp,[-3,15325252384887882227757421748102794318349518712709487389817905929239007568605])

K.<rK> = QuadraticField(-10000006055889179)
O = K.maximal_order()

D = O.discriminant()

print('Discriminant =', D)

m_id = isqrt(p.nbits())+1  #choix fait dans Pegasis

Exemple Sutherland2
Discriminant = -10000006055889179


In [ ]:
prime_max = 370           # taille max polynome modulaire sagemath. 

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

Exemple Sutherland2
Discriminant = -10000006055889179
prime_max : 370
Nombre essais 20
non_friable 1
non_friable 2
non_friable 3
non_friable 4
non_friable 5
non_friable 6
non_friable 7
non_friable 8
non_friable 9
non_friable 10
non_friable 11
non_friable 12
non_friable 13
friable 0
non_friable 14
non_friable 15
non_friable 16
non_friable 17
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 136.65285470485688
degrés moyen de first solution : 549/5
dégrés maximal parmis les solutions : 192
degrés minimal parmis les solutions : 12
Nb idéaux théoriques : 528
Nb idéaux retenus en moyenne : 544
Nb de tentatives ignorées (idéal friable) : 1
Fréquence idéaux friables : 1/21




In [7]:
prime_max = 1000           # taille max polynome modulaire Sutherland  

stat_solutions_clapoti(prime_max, m_id, 20,E,K,O)

prime_max : 1000
Nombre essais 20
non_friable 1
friable 0
friable 1
non_friable 2
friable 2
friable 3
friable 4
friable 5
non_friable 3
non_friable 4
friable 6
non_friable 5
non_friable 6
non_friable 7
friable 7
non_friable 8
non_friable 9
friable 8
friable 9
friable 10
non_friable 10
non_friable 11
non_friable 12
non_friable 13
friable 11
non_friable 14
non_friable 15
non_friable 16
non_friable 17
non_friable 18
non_friable 19
non_friable 20
temps moyen first solution : 94.86853321790696
degrés moyen de first solution : 443/10
dégrés maximal parmis les solutions : 72
degrés minimal parmis les solutions : 12
Nb idéaux théoriques : 528
Nb idéaux retenus en moyenne : 544
Nb de tentatives ignorées (idéal friable) : 12
Fréquence idéaux friables : 3/8




## Ideaux friables et parties friables

In [7]:
def forme_de_norme(l,D):

    #On calcule une forme quadratique de discriminant D et coefficient dominant l. 
    
    if kronecker(D,l) == 1:
        if l == 2 :
            d = Mod(D,8)
            b = square_root_mod_prime_power(d,2,3)
            b = ZZ(b)
        else :
            x = Mod(D,4)
            d = Mod(D,l)
            y = square_root_mod_prime(d,l)
            b = x.crt(y)
            b = ZZ(b)
        ql = BinaryQF([l,b, int(int((b^2 - D))/int(4*l))])
        return ql
    else:
        raise ValueError('l pas split')


def partie_friable(L,Ne):
    
    # On suppose Ne être la partie friable de la norme de L, pour une certaine famille de nombre premier.
    # On calcule la partie friable de l'idéal L. 
    
    # On renvoie deux listes :
    # factoL contenant les idéaux de norme premier apparaissant dans la partie friable, et expoL leurs exposants.
    
    # En particulier on peut utiliser l'algorithme composantes_de_phi du fichier Broker pour calculer 
    # l'isogénie associée à la partie friable de L à partir de factoL et expoL. 
    
    D = O.discriminant()
    qL = L.quadratic_form()
    
    N = L.norm()
    assert N%Ne == 0
    a = qL[0]
    m = N/a    
    assert m.is_square()
    m = isqrt(m)
    
    Ne2 = a.gcd(Ne)
    me = Ne/Ne2
    assert me.is_square()
    me = isqrt(me)
    assert Ne == Ne2*(me^2)
    
    factoL = []
    expoL = []
    if Ne2 > 1:
        facto_Ne2 = Ne2.factor()
        b = qL[1]
        B = O
        for facteur in facto_Ne2:
            p = facteur[0]
            exp = facteur[1]
            qp = forme_de_norme(p,D)
            bp = qp[1]
            Ip = NumberFieldOrderIdeal(O,qp)
            expoL.append(exp)
            for _ in range(exp):
                if Mod(b, 2*p) == Mod(bp, 2*p):
                    B = Ip*B
                    conjug = False
                else:
                    B = (Ip.conjugate())*B
                    conjug = True
            if conjug:
                factoL.append(Ip.conjugate())
            else:
                factoL.append(Ip)

    if me > 1:
        factoL.append(me*O)
        expoL.append(1)

    return factoL, expoL

In [83]:
L = ideal_de_norme(31,O.conductor(),D)
LL = ideal_de_norme(19,O.conductor(),D)
(L).quadratic_form()
partie_friable(19*LL*LL*L,19*19*19*19*31)

([Ideal (7/2*rK + 1/2, 19*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I,
  Ideal (5/2*rK + 1/2, 31*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I,
  Ideal (19/2*rK + 19/2, 19*rK) of Maximal Order generated by 1/2*rK + 1/2 in Number Field in rK with defining polynomial x^2 + 38669866235 with rK = 196646.5515461688?*I],
 [2, 1, 1])

# Test Solution Directe

In [21]:
from sage.rings.finite_rings.integer_mod import square_root_mod_prime
from sage.rings.finite_rings.integer_mod import square_root_mod_prime_power
from sage.rings.number_field.order_ideal import NumberFieldOrderIdeal
import time

def ideal_de_norme(l,f,D):

    #Trouver un idéal de norme l premier.
    #On se place dans un ordre de discriminant D et de conducteur f
    
    if kronecker(D,l) == 1:
        D = Mod(D,l)
        sq = square_root_mod_prime(D,l)
        sm = Mod(-sq,l)
        sq = ZZ(sq)
        sm = ZZ(sm)
        sq = min(sm,sq)
        #print(sq)
        return NumberFieldOrderIdeal(O,[l, -sq + f*rK]) 
    elif kronecker(D,l) == 0:
        return NumberFieldOrderIdeal(O,[l, f*rK])
    else:
        raise "l est inerte"

def forme_de_norme(l,D):

    #On calcule une forme quadratique de discriminant D et coefficient dominant l. 
    
    if kronecker(D,l) == 1:
        if l == 2 :
            d = Mod(D,8)
            b = square_root_mod_prime_power(d,2,3)
            b = ZZ(b)
        else :
            x = Mod(D,4)
            d = Mod(D,l)
            y = square_root_mod_prime(d,l)
            b = x.crt(y)
            b = ZZ(b)
        ql = BinaryQF([l,b, int(int((b^2 - D))/int(4*l))])
        return ql
    else:
        raise ValueError('l pas split')

def solutions_directes(Ls,Le,N,O):
    
    D = O.discriminant()
    
    ql = Ls.quadratic_form()
    L = NumberFieldOrderIdeal(O,(ql.reduced_form()))
    
    B = L*Le
    NB = B.norm()
    qB = B.quadratic_form()
    a = qB[0]
    m = NB/a    
    assert m.is_square()
    m = isqrt(m)
    
    Ne = Le.norm()
    d = a*m
    Borne_candidats = (N*a*Ne)//2
    
    liste_candidats = []
    print('Borne_candidats :', Borne_candidats)
    xmax = isqrt(Borne_candidats)
    for x in range(xmax):
        ymax = (Borne_candidats - x^2)//(-D)
        ymax = isqrt(ymax)
        for y in range(ymax):
            candidat = x^2 - D*y^2
            if candidat < Borne_candidats and candidat % (a*Ne) == 0:
                Nk = candidat//(a*Ne)
                liste_candidats.append([Nk,x,y,d])

    return liste_candidats



In [22]:
#Exemple de Jao "small"

print('Exemple JaoSmall')

p = 10^10 + 19
Fp = GF(p)
E = EllipticCurve(Fp, [15,129])

K.<rK> = QuadraticField(-38669866235)
O = K.maximal_order()                   #Calculer O et K à partir de E ? CF Sutherland, mais pas d'implementation


f = O.conductor()
D = O.discriminant()

print('Discriminant =', D)

l = 5000000029

L = ideal_de_norme(l,f,D)


Exemple JaoSmall
Discriminant = -38669866235


In [23]:
def formes_generatrices (D,N,test_norm): #Trouver la borne : Analyse de Jao

# Trouver une famille génératrice du groupes de classes, vu comme formes quadratiques réduites.
# En n'utilisant que des idéaux de normes premiers autorisés pour le calcul d'isogénies horizontales.
# On génère simplement une famille de N idéaux, N supposé suffisament grand pour que leurs classes soient génératrices.
# N = 2log^2(D) suffit (Admet hypothèse de Riemann Généralisée).
    
    Idéaux = []
    Formes = []
    Primes = []
    i = 2
    while i < N:
        if kronecker(D,i) == 1 and (test_norm)%i != 0:
            Primes.append(i)
            qi = forme_de_norme(i,D)
            Li = NumberFieldOrderIdeal(O,qi)
            Idéaux.append(Li)
            Formes.append(qi)
        i = next_prime(i)
    return Idéaux, Formes, Primes

Idéaux, formes, Primes = formes_generatrices(D,24,p*E.cardinality())

In [24]:
def rand_ideal_friable(Idéaux,O,Borne_t):
    k = len(Idéaux)
    Le = O
    for _ in range(Borne_t):
        i = randint(0,k-1)
        Le = Idéaux[i]*Le
    return Le

Le = rand_ideal_friable(Idéaux,O,7)
Le.norm()

1309856371

In [25]:
N = 1069092 #accessible degré 18

In [27]:
solutions_directes(L,Le,N,O)

Borne_candidats : 84155229675871960249809157374


KeyboardInterrupt: 